# HW1 · Look at the data first

Goal: know the data before modeling it. Answer for the report: How many images per split and class? What do typical images look like? How much do the original sizes and aspect ratios vary (does the center crop cut anything important)? What does the pixel distribution look like (relevant for the likelihood choice)?

Needs the prepared arrays: `uv run python -m dgm.data.chest_xray --sizes 64 128`. Select the `.venv` kernel (top right).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision.utils import make_grid

from dgm.data.chest_xray import CLASSES, ChestXray, describe, find_split_root, list_images, raw_dir

SIZE = 64
print(describe(SIZE))

In [ ]:
# A grid of training images per class
train = ChestXray("train", SIZE)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, label in zip(axes, (0, 1), strict=True):
    idx = (train.labels == label).nonzero().flatten()[:16]
    grid = make_grid(train.as_float()[idx], nrow=4, padding=2, pad_value=1.0)
    ax.imshow(grid[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{CLASSES[label]} ({SIZE}x{SIZE})")
    ax.axis("off")

In [ ]:
# Pixel intensities: are they close to binary (Bernoulli) or spread out (Gaussian)?
x = train.as_float()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(x.flatten().numpy(), bins=100, color="#2a78d6")
axes[0].set_title("pixel values (train)")
axes[1].imshow(x.mean(0)[0], cmap="gray")
axes[1].set_title("mean image")
axes[2].imshow(x.std(0)[0], cmap="magma")
axes[2].set_title("per-pixel std")
for ax in axes[1:]:
    ax.axis("off")

In [ ]:
# Original sizes and aspect ratios (a random sample of raw training files)
def image_size(path):
    with Image.open(path) as img:
        return img.size  # (width, height)


files = [path for path, _ in list_images(find_split_root(raw_dir()), "train")]
rng = np.random.default_rng(0)
sizes = np.array([image_size(files[i]) for i in rng.choice(len(files), 300, replace=False)])
aspect = sizes[:, 0] / sizes[:, 1]
print(f"width  {np.percentile(sizes[:, 0], [5, 50, 95])}")
print(f"height {np.percentile(sizes[:, 1], [5, 50, 95])}")
print(f"aspect ratio (w/h) {np.percentile(aspect, [5, 50, 95]).round(2)}")

## Notes for the report

- Split sizes and class balance:
- What the center crop removes:
- Pixel distribution and what it means for the likelihood:
